Hyperparams selection

# Libraries

In [15]:
import openpyxl
import pandas as pd
from pathlib import Path
import numpy as np
import sys
from tqdm import tqdm
import optuna
import json

In [2]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [3]:
# hyperparams
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import objective

In [4]:
RANDOM_STATE = 42

# Datatasets selection

In [5]:
path_data = Path('../data')
path_results = Path('../results/')
path_highlighted = path_results / 'metrics_modelling2_highlight.xlsx'
df_highlighted = pd.read_excel(path_highlighted)
wb = openpyxl.load_workbook(path_highlighted, data_only=True)
ws = wb.worksheets[0]

In [6]:
highlighted_rows = []
for row in range(1, ws.max_row):
    cell = ws.cell(column=1, row=row)
    bgColor = cell.fill.bgColor.index
    fgColor = cell.fill.fgColor.index
    if bgColor != '00000000':
        highlighted_rows.append(row)
highlighted_rows = np.array(highlighted_rows) - 2
df_filtered = df_highlighted.iloc[highlighted_rows]

Datasets for parameters selection.

In [7]:
df_filtered

,dataset,target,model,accuracy,f1,precision,recall,roc_auc
2,df_modelling_no_multicollinearity,splashing,catboostclassifier_splashing,0.866667,0.900000,0.865385,0.937500,0.839120
3,df_modelling_dimensionless,splashing,kneighborsclassifier_splashing_ordenc,0.866667,0.900000,0.865385,0.937500,0.839120
4,df_modelling_dimensionless,splashing,catboostclassifier_splashing,0.866667,0.900000,0.865385,0.937500,0.839120
5,df_modelling_dimensionless,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.900000,0.931818
6,df_modelling_dimensionless,splashing,kneighborsclassifier_smote_splashing_ordenc,0.866667,0.897959,0.880000,0.916667,0.847222
7,df_modelling_dimensionless,splashing,kneighborsclassifier_splashing_onehot,0.853333,0.891089,0.849057,0.937500,0.820602
8,df_modelling_dimensionless,splashing,svc_smote_splashing_onehot,0.853333,0.888889,0.862745,0.916667,0.828704
11,df_modelling_no_multicollinearity,splashing,catboostclassifier_smote_splashing,0.853333,0.886598,0.877551,0.895833,0.836806
12,df_modelling_dimensionless,splashing,catboostclassifier_smote_splashing,0.853333,0.886598,0.877551,0.895833,0.836806
13,df_modelling_dimensionless,splashing,logisticregression_splashing_onehot,0.840000,0.880000,0.846154,0.916667,0.810185


In [8]:
sorted(df_filtered['model'].unique())

['catboostclassifier_net_impact',
 'catboostclassifier_smote_net_impact',
 'catboostclassifier_smote_splashing',
 'catboostclassifier_splashing',
 'kneighborsclassifier_net_impact_onehot',
 'kneighborsclassifier_net_impact_ordenc',
 'kneighborsclassifier_smote_net_impact_ordenc',
 'kneighborsclassifier_smote_splashing_ordenc',
 'kneighborsclassifier_splashing_onehot',
 'kneighborsclassifier_splashing_ordenc',
 'logisticregression_net_impact_ordenc',
 'logisticregression_splashing_onehot',
 'svc_smote_net_impact_onehot',
 'svc_smote_net_impact_ordenc',
 'svc_smote_splashing_onehot',
 'svc_smote_splashing_ordenc']

# Parameters selection

In [9]:
df_filtered.head(2)

,dataset,target,model,accuracy,f1,precision,recall,roc_auc
2,df_modelling_no_multicollinearity,splashing,catboostclassifier_splashing,0.866667,0.9,0.865385,0.9375,0.83912
3,df_modelling_dimensionless,splashing,kneighborsclassifier_splashing_ordenc,0.866667,0.9,0.865385,0.9375,0.83912


In [16]:
dict_results = {}
for i in (pbar:=tqdm(range(df_filtered.shape[0]))):
    dataset_filename = df_filtered.iloc[i]['dataset']
    if 'df_full' in dataset_filename: continue
    model_str = df_filtered.iloc[i]['model']
    pbar.set_description(f"Processing {model_str}")
    target = df_filtered.iloc[i]['target']
    train, test = get_train_test(
        dataset_filename=dataset_filename,
        target=target)
    study = optuna.create_study(direction="maximize")
    def obj(trial):
        return objective(
            trial, train, test, 
            target, model_str, dataset_filename=dataset_filename, 
            random_state=RANDOM_STATE, cat_features=['wettability'])
    study.optimize(obj, n_trials=3, timeout=10)
    dict_results[model_str] = {
            'dataset': dataset_filename,
            'model': model_str,
            'target': target,
            'f1': study.best_value,
            'best_params': study.best_params}
with open("../results/optuna_results.json", "w") as outfile: 
    json.dump(dict_results, outfile)

Processing catboostclassifier_splashing:   0%|          | 0/22 [00:00<?, ?it/s][I 2024-03-09 12:53:00,946] A new study created in memory with name: no-name-c96c0d69-ca37-4ede-8cc9-218ef2f23d6c
[I 2024-03-09 12:53:02,309] Trial 0 finished with value: 0.9 and parameters: {'objective': 'CrossEntropy', 'colsample_bylevel': 0.06750422065623855, 'depth': 6, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'subsample': 0.7194560675200535}. Best is trial 0 with value: 0.9.
[I 2024-03-09 12:53:06,620] Trial 1 finished with value: 0.8598130841121496 and parameters: {'objective': 'Logloss', 'colsample_bylevel': 0.08317451861599388, 'depth': 6, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'bagging_temperature': 3.3210956082289744}. Best is trial 0 with value: 0.9.
[I 2024-03-09 12:53:07,739] Trial 2 finished with value: 0.8392857142857143 and parameters: {'objective': 'Logloss', 'colsample_bylevel': 0.029529478016267964, 'depth': 9, 'boosting_type': 'Plain', 'bootstrap_type': 